In [4]:
'''# How to run this code in Google Colab or Local Machine: 
#Before running this code, make sure to install the required python and needed packages. 
# Since Iam used to Visual studio code, below instructions are for VSC but can be true fot other similar IDEs.
# If you don't want to install them in Local Machine then you use the colab directly. 
# Choose Runtime -> Python 3(ipykernel)
# If the above is not available, Select another kernel -> Colab -> Auto Conect or New Colab server if you do not have

'''

import sys
from datetime import datetime
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Runtime is running Date and time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))


''' Here we are using the Gdrive to store the file and read it from there. 
If you want to use the local machine or colab folder, you can use the commented code below to read the file.
'''
# If the file is stored in the local machine or in Colab folder, you can use the following code to read the file.
# file_path = r"C:\Users\veera\AI\SampleFile\test-product-review.parquet"
# file_path = r"\sample_data\test-product-review.parquet"

from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted:", os.path.ismount('/content/drive'))

file_path = '/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet'
print("File exists:"+file_path+" -", os.path.isfile(file_path))


# Read the Parquet file 
# with specific columns and filter for English language reviews
# For better understanding I have filtered the data for few columns English language reviews only. You can remove the filter if you want to read all the data.

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

table = pq.read_table( file_path)
# table = pq.read_table(file_path, columns=['review_id','review_body', 'language','stars'],
                    #   filters=[[('language', '==', 'en')]])

#Print the first few rows of the DataFrame to see how the data looks like.
df = table.to_pandas()
df.info()
print(df.head(4))
#print(df.shape)
print("Count of each star rating:"+str(df["stars"].value_counts().to_dict()))

'''DATA CLEANING AND LABEL PREPARATION
Here we will convert the "stars" column to numeric and create a new "sentiment" column based on the star ratings.
We wil keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
We might also need to drop and empty or null values in the review_body"stars" column, as they cannot be used for sentiment analysis. Since Data is clean commenting it out. If you want to use it, you can uncomment the code below.
'''

# df = df[["review_body", "stars"]].dropna().copy()

# Convert stars to numeric as had problems with some non-numeric values in the dataset. Coerce errors to NaN, then drop those rows.
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
# df = df.dropna(subset=["stars"]).reset_index(drop=True)

# Keep sentiment as "positive" for 4 and 5 stars, "negative" for 1 and 2 stars, and "neutral" for 3 stars.
def create_sentiment_label(stars):
    if stars >= 4:
        return "positive"
    elif stars <= 2:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["stars"].apply(create_sentiment_label)

# # Remove neutral examples for a simpler binary sentiment task
# df = df[df["sentiment"] != "neutral"].reset_index(drop=True)

print(df[["stars", "sentiment"]].head(6))
print(df["stars"].value_counts())
print(df["sentiment"].value_counts())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Executable: /usr/bin/python3
Runtime is running Date and time: 2026-10-30 21:58:12
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted: True
File exists:/content/drive/MyDrive/AI/SampleFile/test-product-review.parquet - True
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   review_id         1500 non-null   object
 1   product_id        1500 non-null   object
 2   reviewer_id       1500 non-null   object
 3   stars             1500 non-null   int32 
 4   review_body       1500 non-null   object
 5   review_title      1500 non-null   object
 6   language          1500 non-null   object
 7   product_category  1500 non-null   object
dtypes: int32(1), object(7)
memory usage: 88.0+ KB
    revi

In [7]:
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    DataCollatorWithPadding,
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    Trainer,
    TrainingArguments,
)
!pip install -q optuna
import optuna  # Required for hyperparameter search

# 1. Dataset Setup (Assumes 'df' with 'review_body' and 'sentiment' columns is defined)
# df = pd.DataFrame(data)

# 2. Map string labels to integers (Required for Transformer models)
label_mapping = {"negative": 0, "neutral": 1, "positive": 2}
inv_label_mapping = {0: "negative", 1: "neutral", 2: "positive"}
df["label"] = df["sentiment"].map(label_mapping)

# 3. Train/Test Split (Stratified)
train_df, test_df = train_test_split(
    df, test_size=0.3, random_state=42, stratify=df["label"]
)

# Convert Pandas DataFrames to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# 4. Tokenization using DistilBERT Tokenizer
# model_name = "distilbert-base-uncased"
# tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)
# 4. using XLM-RoBERTa tokenizer for multilingual support
# from transformers import AutoModelForSequenceClassification, AutoTokenizer
# model_name = "xlm-roberta-base"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# 1. Switch to the official Multilingual DistilBERT checkpoint
model_name = "distilbert-base-multilingual-cased"
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

def tokenize_function(examples):
  return tokenizer(
      examples["review_body"], truncation=True, max_length=128
  )


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print(f"Number of training samples: {len(tokenized_train)}")
print(f"Number of test samples: {len(tokenized_test)}")
print("Sample tokenized features:", tokenized_train[0]["input_ids"])


# 5. Model Initialization Function (Required for Optuna hyperparameter search)
def model_init(trial):
  return DistilBertForSequenceClassification.from_pretrained(
      model_name, num_labels=3, id2label=inv_label_mapping, label2id=label_mapping
  )


# 6. Define Evaluation Metric Function
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  preds = np.argmax(predictions, axis=1)
  accuracy = np.mean(preds == labels)
  return {"accuracy": accuracy}


# 7. Configure Training Arguments
training_args = TrainingArguments(
    output_dir="./distilbert_sentiment_results",
    learning_rate=3e-5,
    per_device_train_batch_size=16,  # Parallelize computation
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    # logging_steps=2,
    report_to="none",
    fp16=True,  # 2x speedup via mixed precision
    dataloader_num_workers=2,  # Prevent CPU data bottlenecks
    per_device_eval_batch_size=32,  # <--- Speeds up evaluation steps
)

# 8. Initialize Trainer with model_init
trainer = Trainer(
    model_init=model_init,  # Pass model_init instead of a single model instance
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)


# 9. Define Hyperparameter Search Space for Optuna
def hp_space(trial):
  return {
      "learning_rate": trial.suggest_float(
          "learning_rate", 1e-5, 5e-5, log=True
      ),
      "per_device_train_batch_size": trial.suggest_categorical(
          "per_device_train_batch_size", [4, 8, 16]
      ),
      "num_train_epochs": trial.suggest_int(
          "num_train_epochs", 2, 4
      ),  # Tests between 2 to 4 epochs
      "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
  }


# 10. Run Hyperparameter Tuning
print("Starting Optuna Hyperparameter Search...")
best_run = trainer.hyperparameter_search(
    direction="maximize",  # We want to maximize accuracy
    backend="optuna",
    hp_space=hp_space,
    n_trials=5,  # Number of different hyperparameter combinations to test
)

print("\nBest Hyperparameters Found:")
print(best_run.hyperparameters)

# 11. Apply Best Hyperparameters & Finalize Training
print("\nApplying Best Hyperparameters for Final Training...")
for key, value in best_run.hyperparameters.items():
  setattr(trainer.args, key, value)

trainer.train()

# 12. Evaluate on the Test Set
print("\nEvaluating Best Model on Test Data:")
eval_results = trainer.evaluate(tokenized_test)
print(eval_results)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Evaluating Best Model on Test Data:


Training Loss,Validation Loss,Epoch,Accuracy
0.640435,0.853834,4,0.646667


{'eval_loss': 0.8538340926170349, 'eval_accuracy': 0.6466666666666666}


In [17]:
drive_model_path = "/content/drive/MyDrive/AI/best_distilbert_sentiment_model"
trainer.save_model(drive_model_path)
tokenizer.save_pretrained(drive_model_path)
print("Model safely saved to your Google Drive!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model safely saved to your Google Drive!
